# LFM Tiling Pipeline Examples
This notebook has examples on how to use the tiling pipeline included in the LFM repo. 

## Purpose of this notebook
This notebook is designed to get you familiar with the different types of tiling workflows; 
- Query by LTM zone and tile index
- Query by lat, lon point
- Query by bounding box in lat, lon coords (upper left and lower right coordinates)

All cells use an imageDir, which represents the specific input data used for that example. 

## Getting started
You can get started with this notebook by downloading it with:

```bash
wget https://raw.githubusercontent.com/nasa-nccs-hpda/lfm/refs/heads/main/notebooks/run_tiling.ipynb
```

## Imports, clone Git repo

In [1]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import sys
from typing import List
from osgeo import gdal
import subprocess
import warnings

<jemalloc>: Unsupported system page size


In [2]:
# Configuration
repo_dir = "lfm"
repo_url = f"https://github.com/nasa-nccs-hpda/{repo_dir}"

# Try to clone/pull repository (non-critical)
try:
    if not os.path.exists(repo_dir):
        subprocess.run(
            ["git", "clone", repo_url],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"✓ Successfully cloned repository to {repo_dir}/")
    else:
        subprocess.run(
            ["git", "-C", repo_dir, "pull"],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"✓ Successfully pulled latest changes in {repo_dir}/")
except (subprocess.CalledProcessError, FileNotFoundError) as e:
    warnings.warn(
        f"Could not clone/pull repository (this is a known issue in Jupyter). "
        f"If imports fail, run in terminal: git clone {repo_url}",
        UserWarning
    )

# Add to path if exists
if os.path.exists(repo_dir):
    repoPath = Path(os.getcwd()) / repo_dir
    sys.path.append(str(repoPath.parent))

# Import required modules (critical - will raise ImportError if fails)
try:
    from lfm.model.Pipeline import Pipeline
    print("✓ Successfully imported LFM modules")
except ImportError as e:
    print(f"❌ ERROR: Failed to import required modules from {repo_dir}/")
    print(f"   {e}")
    print(f"\nPlease ensure the repository exists by running in your terminal:")
    print(f"  git clone {repo_url}")
    raise  # Re-raise to stop execution

/tmp/ipykernel_3867715/1773854852.py:24: UserWarning: Could not clone/pull repository (this is a known issue in Jupyter). If imports fail, run in terminal: git clone https://github.com/nasa-nccs-hpda/lfm
  warnings.warn(


✓ Successfully imported LFM modules


## User configuration
`outDir`: where you want the tiling outputs to go
`lfmDir`: this is the base directory data will be pulled from. If you modify this, the paths below will be changed. Ensure the path contains the data you want, and look at each example's outDir subdirectory.

In [3]:
outDir = Path("/explore/nobackup/projects/lfm/tiling_outputs")
outDir.mkdir(exist_ok=True, parents=True)

lfmDir = Path("/explore/nobackup/projects/lfm")

## Helper function
This function helps retrieve a .shp file used by the pipeline, in the same directory as the images. 

In [4]:
MOON_SRS = (
    'GEOGCRS["Moon (2015) - Sphere / Ocentric", '
    'DATUM["Moon (2015) - Sphere", '
    'ELLIPSOID["Moon (2015) - Sphere",1737400,0, '
    'LENGTHUNIT["metre",1]]], '
    'PRIMEM["Reference Meridian",0, '
    'ANGLEUNIT["degree",0.0174532925199433]], '
    'CS[ellipsoidal,2], '
    'AXIS["geodetic latitude (Lat)",north, ORDER[1], '
    'ANGLEUNIT["degree",0.0174532925199433]], '
    'AXIS["geodetic longitude (Lon)",east, ORDER[2], '
    'ANGLEUNIT["degree",0.0174532925199433]], '
    'ID["IAU",30100,2015], '
    'REMARK["Source of IAU Coordinate systems: https://doi.org/10.1007/s10569-017-9805-5"]]'
)    

# ----------------------------------------------------------------------------
# getTileDbPath
# ----------------------------------------------------------------------------
def getTileDbPath(imageDir: Path, dbName: str = 'output_index.shp') -> Path:

    if not imageDir or not imageDir.exists() or not imageDir.is_dir():
        raise ValueError('A valid image directory must be provided.')

    fullPath: Path = imageDir / dbName

    if fullPath.exists():
        return fullPath

    outFile: Path = imageDir / dbName
    
    # The database does not exist, so create it for the image directory.
    gdal.TileIndex(outFile, list(imageDir.glob('*.tif')), outputSRS=MOON_SRS)
    outFile.chmod(666)
    return outFile

## Example 1: LTM zone/tile index query for static data
The first cell runs for the Diviner data, while the second runs for the gravity data. 

### Variables
- `imageDir`: input data directory
- `zone`: LTM zone, number and hemisphere
- `zoom level`: Armstrong zoom level, corresponds to tile size and number of tiles
- `tileDbPath`: path to tile database, fetched from helper function
- `pipeline`: instance of the tiling pipeline class
- `cubeFile{x}`: output of pipelines run on different tile indices

In [5]:
imageDir = lfmDir / 'processed_data/Lunar/Static_final/Diviner/Hpar/'  # input image directory
outDirDiviner = outDir / "Diviner/Hpar"  # create a subdir for this example
outDirDiviner.mkdir(exist_ok=True, parents=True)
zone = '37S'  # LTM zone
zoomLevel = 5  # zoom level

print(imageDir)

# Run the pipeline for a given tile index
tileDbPath = getTileDbPath(imageDir)  # .shp database of inputs
pipeline = Pipeline(tileDbPath, outDirDiviner)  # tiling pipeline
cubeFiles = pipeline.runTileIndex(0, 0, zone, zoomLevel)  # run pipeline for tile index (0, 0)
print('First 10 cube files:', cubeFiles[:10])  # print output filenames

/explore/nobackup/projects/lfm/processed_data/Lunar/Static_final/Diviner/Hpar
Processing tile (0, 0) / zone 37S / zoom 5
Total WAC product IDs: 1


AttributeError: 'NoneType' object has no attribute 'SetSpatialFilterRect'

## Example 2: Point query for NAC/WAC
For the NAC and WAC data, a point 149.8 deg E/1.2 deg N was given. This translates to tile(1, 63).

### Variables
- `imageDir`: input data directory
- `zone`: LTM zone, number and hemisphere
- `zoom level`: Armstrong zoom level, corresponds to tile size and number of tiles
- `tileDbPath`: path to tile database, fetched from helper function
- `pipeline`: instance of the tiling pipeline class
- `cubeFile{x}`: output of pipelines run on different tile indices

In [ ]:
imageDir = lfmDir / "processed_data/Lunar/LRO_NAC_Pho_Sites"  # input image directory
outDirNACProcessed = outDir / "NAC_Processed"  # create a subdir for this example
outDirNACProcessed.mkdir(exist_ok=True, parents=True)
zone = '42N'  # LTM zone
zoomLevel = 5  # zoom level

lon = 149.8  # query lat
lat = 1.2  # query lon

tileDbPath = getTileDbPath(imageDir)  # .shp database of inputs
pipeline = Pipeline(tileDbPath, outDirNACProcessed)  # tiling pipeline
cubeFiles: Path = pipeline.runPoint(lat, lon, zone, zoomLevel)  # run pipeline for lat, lon point
print('First 10 cube files:', cubeFile[:10])  # print output filenames

In [ ]:
imageDir = lfmDir / "processed_data/Lunar/LRO_WAC_Pho_Sites"  # input image directory
outDirWACProcessed = outDir / "WAC_Processed"  # create a subdir for this example
outDirWACProcessed.mkdir(exist_ok=True, parents=True)
zone = '42N'  # LTM zone
zoomLevel = 5  # zoom level

lon = 149.8  # query lat
lat = 1.2  # query lon

dbPath = getTileDbPath(imageDir)  # .shp database of inputs
pipeline = Pipeline(dbPath, outDirWACProcessed)  # tiling pipeline
cubeFiles = pipeline.runPoint(lat, lon, zone, zoomLevel)  # run pipeline for lat/lon point
print('First 10 cube files:', cubeFiles[:10])  # print output filenames

## Example 3: AOI Query for NAC/WAC
Instead of providing a tile index, an area of interest can be used.  This discovers all tiles intersecting the area and processes them.

For the NAC and WAC data, a point 149.8 deg E/1.2 deg N was previously given.  We arbitrarily chose an AoI around this.
- Upper left: (149.7, 1.3)
- Lower right: (149.9, 1.1)

### Variables
- `imageDir`: input data directory
- `zone`: LTM zone, number and hemisphere
- `zoom level`: Armstrong zoom level, corresponds to tile size and number of tiles
- `tileDbPath`: path to tile database, fetched from helper function
- `pipeline`: instance of the tiling pipeline class
- `cubeFile{x}`: output of pipelines run on different tile indices

In [ ]:
imageDir = lfmDir / "processed_data/Lunar/LRO_NAC_Pho_Sites" # input image directory
outDirAOINAC = outDir / "NAC_Processed_AOI"  # create a subdir for this example
outDirAOINAC.mkdir(exist_ok=True, parents=True)

# Upper-left and lower-right lat/lon points for bounding box
ulLat = 1.3
ulLon = 149.7
lrLat = 1.1
lrLon = 149.9
zoomLevel = 1  # zoom level 1 for this simple example

tileDbPath = getTileDbPath(imageDir)  # .shp database of inputs
pipeline = Pipeline(tileDbPath, outDirAOINAC)  # tiling pipeline

# run pipeline for AOI
cubeFiles: List[Path] = pipeline.run(ulLat,
                                     ulLon,
                                     lrLat,
                                     lrLon,
                                     zoomLevel)

print('First 10 cube files:', cubeFiles[:10])  # print output filenames

In [ ]:
imageDir = lfmDir / "processed_data/Lunar/LRO_WAC_Pho_Sites"
outDirAOIWAC = outDir / "WAC_Processed_AOI"
outDirAOIWAC.mkdir(exist_ok=True, parents=True)

# Upper-left and lower-right lat/lon points for bounding box
ulLat = 31.241501479374616
ulLon = 121.01616204071722
lrLat = 30.201163639632075
lrLon = 122.22643208573085
zoomLevel = 5 # zoom level 5 for this sub-example

tileDbPath = getTileDbPath(imageDir)  # .shp database of inputs
pipeline = Pipeline(tileDbPath, outDirAOIWAC)  # tiling pipeline

# run pipeline for AOI
cubeFiles: List[Path] = pipeline.run(ulLat,
                                     ulLon,
                                     lrLat,
                                     lrLon,
                                     zoomLevel)

print('First 10 cube files:', cubeFiles[:10])  # print output filenames